In [8]:
import pandas as pd
import numpy as np
import itertools

MutliFingerFlexion Adaptation:
Having participants produce finger extension or flexion of index, ring, or both. In each block, one condition is produced (n_trials = 40). A random cursor purturbation with zero mean and a certain variance will be added. Chords are chosen randomly. They should flex at -2 N or extend at +2N.

In [9]:
fingers = [2, 4]

one_finger_chords = [[f] for f in fingers]
two_finger_chords = list(itertools.combinations(fingers, 2))
chords = one_finger_chords + two_finger_chords

ext_flex_amp = [-2, 2]

iti = 1000 # msecs for inter-trial interval
planTime = 2000 # msecs for precue time
feedbackTime = 2000 # msecs for feedback time

purturbation_std = 0.5 # Newton


total_sub_num = 2
num_trials_per_block = 30


In [10]:
def chord_forces(chord, target_force):
    return [target_force if i in list(chord) else 0 for i in range(1,6)]

In [14]:
### target file headers: [subNum, planTime, feedbackTime, iti, session, 
# targetForce1, targetForce2, targetForce3, targetForce4, targetForce5]

for sub in range(total_sub_num):

    subNum = sub + 1
    bn = 1
    # shuffle chords
    np.random.seed(sub)
    chord_order = chords.copy()
    np.random.shuffle(chord_order)
    for chord in chord_order:
        target_forces = np.zeros(5)
        for target_force1 in ext_flex_amp:
            if len(chord) == 1:
                seed = hash((tuple(chord), target_force1, 0)) % (2**32 -1)    
                np.random.seed(seed)        
                target_forces[chord[0]-1] = target_force1
                # purturbation1 = index (finger 2), purturbation2 = ring (finger 4)
                noise = np.round(np.random.normal(0, purturbation_std, num_trials_per_block), 2)
                if chord[0] == 2:
                    purturbation1, purturbation2 = noise, 0
                else:
                    purturbation1, purturbation2 = 0, noise

                block_data = pd.DataFrame([[subNum, target_forces[0], target_forces[1], target_forces[2], target_forces[3], target_forces[4], 0,
                            0, 0, planTime, feedbackTime, iti]], 
                            columns = ['subNum', 'targetForce1', 'targetForce2', 'targetForce3', 'targetForce4', 'targetForce5', 'isTargetVisible',
                                    'purturbation1', 'purturbation2', 'planTime', 'feedbackTime', 'iti'])
                block_data = pd.concat([block_data]*num_trials_per_block, ignore_index=True)

                for block_type in ['unpurturbed', 'purturbed', 'no_feedback']:

                    block_data = pd.DataFrame([[subNum, target_forces[0], target_forces[1], target_forces[2], target_forces[3], target_forces[4], 0,
                            0, 0, planTime, feedbackTime, iti]], 
                            columns = ['subNum', 'targetForce1', 'targetForce2', 'targetForce3', 'targetForce4', 'targetForce5', 'isTargetVisible',
                                    'purturbation1', 'purturbation2', 'planTime', 'feedbackTime', 'iti'])

                    block_data = pd.concat([block_data]*num_trials_per_block, ignore_index=True)
                    if block_type == 'unpurturbed':
                        block_data['isTargetVisible'] = 1
                    elif block_type == 'purturbed':
                        block_data['purturbation1'] = purturbation1
                        block_data['purturbation2'] = purturbation2
                        block_data['isTargetVisible'] = 1
                    elif block_type == 'no_feedback':
                        block_data['isTargetVisible'] = 0

                    block_data.to_csv(f'sn{subNum}_{bn}.tgt', index=False, sep='\t')
                    bn += 1

            elif len(chord) == 2:
                for target_force2 in ext_flex_amp:
                    seed = hash((tuple(chord), target_force1, target_force2)) % (2**32 -1)      
                    np.random.seed(seed)
                    target_forces[chord[0]-1] = target_force1
                    target_forces[chord[1]-1] = target_force2
                    purturbation1 = np.round(np.random.normal(0, purturbation_std, num_trials_per_block),2)
                    purturbation2 = np.round(np.random.normal(0, purturbation_std, num_trials_per_block),2)


                    for block_type in ['unpurturbed', 'purturbed', 'no_feedback']: 
                        block_data = pd.DataFrame([[subNum, target_forces[0], target_forces[1], target_forces[2], target_forces[3], target_forces[4], 0,
                            0, 0, planTime, feedbackTime, iti]], 
                            columns = ['subNum', 'targetForce1', 'targetForce2', 'targetForce3', 'targetForce4', 'targetForce5', 'isTargetVisible',
                                    'purturbation1', 'purturbation2', 'planTime', 'feedbackTime', 'iti'])
                        block_data = pd.concat([block_data]*num_trials_per_block, ignore_index=True) 
                        if block_type == 'unpurturbed':
                            block_data['isTargetVisible'] = 1
                        elif block_type == 'purturbed':
                            block_data['purturbation1'] = purturbation1
                            block_data['purturbation2'] = purturbation2
                            block_data['isTargetVisible'] = 1
                        elif block_type == 'no_feedback':
                            block_data['isTargetVisible'] = 0

                        block_data.to_csv(f'sn{subNum}_{bn}.tgt', index=False, sep='\t')
                        bn += 1
    
    